# S2 — in-silico visual localizer (TRIBE v2)

Runs the **frozen** Phase C design. Every parameter comes from `neurocheck/s2_design.py`.

**Order matters. Do not skip ahead.**

If step 3 says **INDEX-based**, stop. Do not run step 5. The stimulus must be
re-rendered at 16 fps on a machine you control, then re-uploaded.

## 1 · Setup

In [ ]:
BRANCH = "main"
DATASET = None          # auto-detected; override with an explicit path if needed

import subprocess, sys, os, glob
from pathlib import Path

REPO = "/kaggle/working/tribe-bench"
if not Path(REPO).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "50",
                    "https://github.com/codesbydevesh/tribe-bench.git", REPO], check=True)
os.chdir(REPO)
print("HEAD:", subprocess.run(["git", "log", "--oneline", "-1"],
                              capture_output=True, text=True).stdout.strip())

# Search ANY depth: Kaggle mounts at /kaggle/input/datasets/<user>/<slug>/ on some
# accounts and /kaggle/input/<slug>/ on others. Anchor on the files, not the layout.
if DATASET is None:
    for mp4 in sorted(glob.glob("/kaggle/input/**/s2_stimulus.mp4", recursive=True)):
        root = str(Path(mp4).parent)
        if (Path(root) / "floc").is_dir():
            DATASET = root
            break

print("stimulus root:", DATASET or "NOT FOUND")
if DATASET is None:
    print("\ncontents of /kaggle/input:")
    for q in sorted(glob.glob("/kaggle/input/**", recursive=True))[:40]:
        print("  ", q)
assert DATASET, "no attached dataset contains both floc/ and s2_stimulus.mp4"
os.environ["S2_STIMULUS_ROOT"] = DATASET
print("floc images:", len(glob.glob(DATASET + "/floc/*/*.jpg")))

In [ ]:
!pip install -q -e . 2>&1 | tail -2
!pip install -q git+https://github.com/facebookresearch/tribev2.git 2>&1 | tail -3

## 2 · Verify the uploaded inputs

CPU-only, read-only. **8/8 must pass.** Anything else means re-upload.

In [ ]:
!python3 scripts/s2_verify_inputs.py --stimulus-root $S2_STIMULUS_ROOT

## 3 · Frame-sampling check — the decision point

Does `neuralset` pick V-JEPA's 64 frames by **timestamp** or by **frame index**?

* timestamp -> 8 fps is fine, continue
* index -> 64 frames at 8 fps span **8 s instead of 4**. **STOP.**

In [ ]:
rc = subprocess.run([sys.executable, "scripts/s2_check_frame_sampling.py"]).returncode
FRAME_OK = (rc == 0)
print({0: "TIMESTAMP - safe to continue",
       1: "INDEX - STOP. Re-render at 16 fps locally, re-upload.",
       2: "AMBIGUOUS - resolve by hand before running."}.get(rc, "rc=%d" % rc))

## 4 · Go / no-go

In [ ]:
assert FRAME_OK, "frame-sampling check did not pass - do not run S2"
!python3 scripts/s2_go_no_go.py --review-clean --neuralset-timestamp

## 5 · Run S2

One forward pass over the 1050 s stimulus. Both lags are scored from the same
timecourse. Budget ~3.4 h as an upper bound.

In [ ]:
assert FRAME_OK, "frame-sampling check did not pass - do not run S2"
!python3 scripts/s2_run.py --infer --stimulus-root $S2_STIMULUS_ROOT

## 6 · Compliance

In [ ]:
!python3 scripts/s2_check_compliance.py data/s2_report.json

## 7 · Save the outputs

In [ ]:
import shutil
for f in ("data/s2_report.json", "data/s2_manifest.json"):
    if Path(f).exists():
        shutil.copy(f, "/kaggle/working/" + Path(f).name)
        print("saved", Path(f).name)